This is a Jupyter notebook to perform collision cross-section prediction using GraphCCS [Large-scale prediction of collision cross-section with very deep graph convolutional network for small molecule identification, Xie et al. (https://doi.org/10.1016/j.chemolab.2024.105177)], with uncertainty guarantees using myopic MCES distances [Coverage bias in small molecule machine learning, Kretschmer et al. (https://doi.org/10.1038/s41467-024-55462-w)] as the basis for out-of-distribution detection [Testing for outliers with conformal *p*-values, Bates et al. (https://doi.org/10.1214/22-AOS2244)] and adaptive predictive distributions using conformal predictive sytems [Computationally efficient versions of conformal predictive distributions, Vovk et al. (https://doi.org/10.1016/j.neucom.2019.10.110) & Mondrian Conformal Predictive Distributions, Boström et al. (https://proceedings.mlr.press/v152/bostrom21a.html)].

The input is a .tsv file named _input.tsv_, which must be place in the same folder as this Jupyter notebook. It must contain the following columns (the order does not matter; neither if there is other user-specific information):
- SMILES strings (column name: *SMILES*) and/or
- InChI strings (column name: *InChI*). Having both types of molecular identifiers can help in case the molecular structure standardization process fails with SMILES strings, which are given priority.
- Ion type strings in the following format: [M+H]+, [M+Na]+ and [M-H]-; other ion types are not supported (column name: *Ion Type*).
- Empirical CCS of the NTS feature of each candidate structure (column name: *CCS*)

The output is another .tsv file named _output.tsv_, which will be placed in the same folder. It is structured in the same way as _input.tsv_ with the following additional columns:
- *P^OOD*: probability for the candidate structure to be out-of-distribution
- *CCShat*: predicted CCS for the candidate structure
- *P^CCS*: probability for the empirical CCS of the corresponding NTS feature to originate from the predictive distribution of a candidate structure
- *CRPS*: continuous ranked probability score. It quantifies how close and how tight the predictive distribution of a candidate structure is around the CCS of the corresponding NTS feature

The parent folder must contain:
- Scripts for standardizing molecular structures and calculating myopic MCES distances (_structures.py_)
- Molecular graphs of the training molecules used to calculate myopic MCES distances (*background_graphs*)
- GraphCCS Python script (_GraphCCS.py_)
- GraphCCS PyTorch .pt file (_GraphCCS.pt_)
- Python scripts for out-of-distribution detection and conformal predictive systems (_OODDS.py_, _CPS.py_)
- .tsv file (_cal_data.tsv_) of calibration data with three columns:
    - _MCES_: calibration arithmetic means of myopic MCES distances
    - _Nonconformity_: calibration non-conformity scores for the conformal predictive system
    - _Ion Type_: calibration ion types

The notebook requires two installed Python environments to run:
1. The environment to perform molecular structure standardization and myopic MCES distance calculations
(https://github.com/AlBi-HHU/myopic-mces)
2. The environment to perform CCS prediction and apply the conformal predictive system
(https://github.com/tingxiecsu/GraphCCS)

The places where each environment should be selected are <mark>highlighted in yellow</mark>. As it is, the notebook does not allow to automatically shut down one environment and start the next one. 

# 1. **Molecular structure standardization and myopic MCES distance calculations**

<mark> Select environment for myopic MCES distance calculations

In [ ]:
# environment: myopic MCES distance


import os
import warnings

import numpy as np
import pandas as pd

from structures import MOLstandardizer
from rdkit import Chem

from structures import construct_graph, myopicMCESruler
import networkx as nx

In [ ]:
input_tsv = pd.read_csv('input.tsv', header = 0, sep='\t')

In [ ]:
                             ######################################################
# -------------------------- ######## molecular structure standardization ######### --------------------------
                             ######################################################

In [ ]:
def robustMolFromSmiles(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None
    
def robustMolFromInchi(inchi):
    try:
        return Chem.MolFromInchi(inchi)
    except Exception:
        return None
    
def robustMolToSmiles(mol):
    try:
        return Chem.MolToSmiles(mol)
    except Exception:
        return None

In [ ]:
if 'SMILES' in input_tsv.columns: # preference for SMILES identifiers
    print("Processing SMILES strings (20000 in ~2 min).")
    standardized_id = pd.DataFrame(
        input_tsv.SMILES.apply(MOLstandardizer, input_type='SMILES').to_list()
    )
elif 'InChI' in input_tsv.columns:
    print("Processing InChI strings (20000 in ~2 min).")
    standardized_id = pd.DataFrame(
        input_tsv.InChI.apply(MOLstandardizer, input_type='InChI').to_list()
    )
else:
    raise ValueError(
        'At least either SMILES or InChI strings must be provided, ' \
        'with corresponding input.csv columns "SMILES" and "InChI", respectively.'
    )

In [ ]:
fails1 = standardized_id.loc[standardized_id.isna().all(axis=1)].index.to_numpy()
if fails1.size == 0:
    print('All molecular structures succesfully converted.')
else:
    if 'SMILES' in input_tsv.columns and 'InChI' in input_tsv.columns: # try with InChI to solve the errors
        print(
            'Some SMILES strings failed conversion when standardizing the molecular structure; trying standardization with InChI strings...'
        )
        corrections = pd.DataFrame(
            input_tsv.loc[fails1, 'InChI'].apply(lambda x: MOLstandardizer(x, input_type='InChI')).to_list(), index = fails1
        )
    
        fails2 = corrections.loc[corrections.isna().all(axis=1)].index.to_numpy()
        corrected_inputs = np.setdiff1d(fails1, fails2)
        standardized_id.loc[corrected_inputs] = corrections.loc[corrected_inputs]
        if fails2.size > 0:
            # manual checking is required
            # otherwise, functions in later stages will not convert the molecular identifiers to a rdkit.Chem.rdchem.Mol object
            raise Exception(
                'Some molecular identifiers failed conversion when standardizing both from SMILES and InChI strings. ' \
                'Manual handling is required. Use the indexes contained in "fails2" to check the failed instances in input.tsv.'
            )
        else:
            print('All secondary InChI strings succesfully converted.')

    else:
        # only one type of molecular identifier was given and manual checking is required
        # otherwise, functions in later stages will not convert the molecular identifiers to a rdkit.Chem.rdchem.Mol object
        raise Exception(
            'Some molecular identifiers failed conversion when standardizing.' \
            'Manual handling is required. Use the indexes contained in "fails1" to check the failed instances in input.tsv.'
        )

In [ ]:
# double-check that the standardized molecular identifiers can be converted back to rdkit.Chem.rdchem.Mol objects
# sometimes it happens that this is not the case; fall back then to the original input identifier
print("Double-checking standardized molecular identifiers by converting them back to rdkit.Chem.rdchem.Mol objects.")
fails3 = standardized_id.loc[
    standardized_id['Absolute SMILES'].apply(robustMolFromSmiles).isna()
].index.to_numpy()

if fails3.size == 0:
    print('Standardization succesful.')

else:
    # try the InChI strings for the failed conversions from SMILES
    fails3 = fails3[
        standardized_id.loc[fails3, 'InChI'].apply(robustMolFromInchi).isna().to_numpy()
    ]

    if fails3.size == 0:
        print('Standardization succesful.')

    else:
        warnings.warn(
            'Some molecular identifiers could not be converted back to rdkit.Chem.rdchem.Mol objects; ' \
            'falling back to original molecular identifiers.'
        )

        if 'SMILES' in input_tsv.columns:

            if 'InChI' in input_tsv.columns:
                # fails3 errors coming from SMILES strings standardization
                smiles_fails = np.intersect1d(fails3, np.setdiff1d(np.arange(input_tsv.shape[0]), fails1))
                standardized_id.loc[smiles_fails, 'Absolute SMILES'] = input_tsv.loc[smiles_fails, 'SMILES']
                standardized_id.loc[smiles_fails, 'InChI'] = None
                
                # fails3 errors coming from InChI strings
                inchi_fails = np.intersect1d(fails3, fails1)
                standardized_id.loc[inchi_fails, 'InChI'] = input_tsv.loc[inchi_fails, 'InChI']
                standardized_id.loc[inchi_fails, 'Absolute SMILES'] = None

                
            else:
                standardized_id.loc[fails3, 'Absolute SMILES'] = input_tsv.loc[fails3, 'SMILES']
                standardized_id.loc[fails3, 'InChI'] = None

        else:
            standardized_id.loc[fails3, 'InChI'] = input_tsv.loc[fails3, 'InChI']
            standardized_id.loc[fails3, 'Absolute SMILES'] = None

In [ ]:
# save (semi-)standardized molecular identifiers for CCS prediction later
standardized_id.to_csv('IDs.tsv', index=False, sep='\t')

In [ ]:
                             ######################################################
# -------------------------- ############### myopic MCES distances ################ --------------------------
                             ######################################################

In [ ]:
# read in the background graphs
print('Reading background graphs (~2/3 min).')
background_graphs = [nx.read_gml(os.path.join("background_graphs", f)) for f in os.listdir("background_graphs")]


# construct the graphs for the queries
print('Constructing graphs for myopic MCES distance calculations.')
graphs = [construct_graph(mol_identifiers) for _, mol_identifiers in standardized_id.iterrows()]

# conversions that failed
graph_fails = [i for i, g in enumerate(graphs) if g is None]
if len(graph_fails) > 0:
    raise Exception(
        'Some molecular identifiers could not be converted to a networkx.classes.graph.Graph object.' \
        'Check the specific entries using the indexes in "graph_fails".'
    )
else:
    print('Graph conversion succesful.')

In [ ]:
# 2000 chemicals on 10 CPUs takes ~90 minutes (depends on what molecules are in input.tsv)
# n_jobs set to all available CPUs by default, change if needed
avg_distances = myopicMCESruler(graphs, background_graphs, threshold = 10, n_jobs = -1, k = 46)

MCESfails = np.isnan(avg_distances)
if np.sum(MCESfails) > 0:
    MCESfails = np.where(MCESfails)[0] # obtain indexes of failed instances
    raise Exception(
        'Some instances failed myopic MCES arithmetic mean calculation.\
        Check the specific entries using the indexes in "MCESfails".'
    )
else:
    np.save('myopicMCESavg.npy', avg_distances) # save arithmetic means for later stages
    print('MCES distance calculation succesful.')

# 2. **CCS prediction and conformal predictive system**

<mark> Select environment to perform CCS prediction and apply the conformal predictive system

In [ ]:
# environment: GraphCCS


import os
import warnings

from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
if not torch.cuda.is_available():
    warnings.warn(
        'GPU not available. This will result in slower CCS prediction.'
    )
from GraphCCS import ion2graph, loadGraphCCS, predict

from CPS import ConformalPredictiveSystem
from OODDS import OODDS

In [ ]:
input_tsv = pd.read_csv('input.tsv', header = 0, sep='\t')

In [ ]:
ion_types = input_tsv['Ion Type'].to_numpy()
unsupported_ion_types = ~np.isin(ion_types, ['[M+H]+', '[M+Na]+', '[M-H]-'])
if unsupported_ion_types.any():
    unsupported_ion_types = np.where(unsupported_ion_types)[0]
    raise ValueError(
        'Some ion types are not supported. ' \
        'Use the indexes contained in "unsupported_ion_types" to check the invalid instances.'
    )

In [ ]:
results = {}

In [ ]:
                             ######################################################
# -------------------------- ############### CCS point estimation ################# --------------------------
                             ######################################################

In [ ]:
# calculate ion graphs
molecular_identifiers = pd.read_csv('IDs.tsv', header=0, sep='\t')
ions = pd.DataFrame({'Absolute SMILES': molecular_identifiers['Absolute SMILES'], 'InChI': molecular_identifiers['InChI'], 'Ion Type': ion_types})
ion_graphs = [ion2graph(ion) for _, ion in tqdm(ions.iterrows(), desc='Calculating ion graphs', unit= ' Graphs', total = input_tsv.shape[0])]

# conversions that failed
ion_graph_fails = [i for i, g in enumerate(ion_graphs) if g is None]
if len(ion_graph_fails) > 0:
    raise Exception(
        'Some molecular identifiers could not be converted to a networkx.classes.graph.Graph object.' \
        'Check the specific entries using the indexes in "ion_graph_fails".'
    )
else:
    print('Ion graph conversion succesful.')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = {
    'batch_size': 64,
    'num_workers': 0,
    'num_layers': 40,
    'hid_dim': 400,
    'gru_out_layer': 2,
    'edge_feat_size': 18,
    'node_feat_size': 148
}

In [ ]:
# load GraphCCS model
model = loadGraphCCS(
    GraphCCSconfig = config, 
    weights_path = 'GraphCCS.pt',
    device = device
)

In [ ]:
# point estimation
CCShat = predict(
    GraphCCSconfig = config,
    ions = ion_graphs,
    model = model,
    device = device
)

CCShat_fails = np.isnan(CCShat)
if np.sum(CCShat_fails) > 0:
    CCShat_fails = np.where(CCShat_fails)[0] # obtain indexes of failed instances
    raise Exception('Some instances failed CCS prediction.\
        Check the specific entries using the indexes in the array "CCShat_fails".')
else:
    print('CCS prediction succesful.')

In [ ]:
                             ######################################################
# -------------------------- ############ conformal predictive system ############# --------------------------
                             ######################################################

In [ ]:
cal_data = pd.read_csv('cal_data.tsv', sep='\t', header = 0)
cal_mces = cal_data.MCES.to_numpy()
nonconformities = cal_data.Nonconformity.to_numpy()
cal_ion_types = cal_data['Ion Type'].to_numpy()

mces = np.load('myopicMCESavg.npy')
CCS = input_tsv.CCS.to_numpy()

In [ ]:
# out-of-distribution detection
oodds = OODDS()
oodds.fit(cal_mces)
results['P^OOD'] = oodds.getOODps(mces)

In [ ]:
# assemble conformal predictive system
cps = ConformalPredictiveSystem()
cps.nonconformities = nonconformities / (cal_mces ** 2)
cps.Mondrian(cal_taxonomies = cal_ion_types, ccv = True, seed = 42) # change seed for randomness
print('Conformal predictive system assembled.')

In [ ]:
# calculate P^CCS
print('Calculating P^CCS.')
PCCS = cps.getPvalues(
    y = CCS, y_hat = CCShat, difficulties = mces ** 2, taxonomies = ion_types, seed = 42 # change seed for randomness
)
PCCSfails = np.isnan(PCCS)
if np.sum(PCCSfails) > 0:
    PCCSfails = np.where(PCCSfails)[0] # obtain indexes of failed instances
    raise Exception('Some instances failed calculation of P^CCS.\
        Check the specific entries using the indexes in the array "PCCSfails".')
else:
    print('Calculation of P^CCS succesful.')
    results['CCShat'] = CCShat
    results['P^CCS'] = PCCS

In [ ]:
# calculate CRPS
print('Calculating CRPSs.')
CRPS = cps.getCRPSs(
    y = CCS, y_hat = CCShat, difficulties = mces ** 2, taxonomies = ion_types, verbose = False
)
CRPSfails = np.isnan(CRPS)
if np.sum(CRPSfails) > 0:
    CRPSfails = np.where(CRPSfails)[0] # obtain indexes of failed instances
    raise Exception('Some instances failed calculation of CRPS.\
        Check the specific entries using the indexes in the array "CRPSfails".')
else:
    print('Calculation of CRPS succesful.')
    results['CRPS'] = CRPS

In [ ]:
pd.concat([input_tsv, pd.DataFrame(results)], axis=1).to_csv('output.tsv', index=False, sep='\t')

# 3. **Clean temporary files**

In [ ]:
os.remove('IDs.tsv')
os.remove('myopicMCESavg.npy')